# 02 · Dataset metadata and validation

Build Phase 2 metadata for COVIDx CXR-3A version 7 and audit locally accessible images. This notebook preserves the official train/test partition and does **not** create final training splits. It does not download images automatically or claim that unavailable files were validated.

## 1. Configure permitted data access

Review `docs/DATASET.md` and accept the required source terms first. In Colab, mount Google Drive if the licensed version-7 archive is already stored there. You may instead set `COVIDX_DATA_DIR` and `COVIDX_LABEL_DIR` to local paths. Keep all images, metadata CSV files, and credentials outside Git.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Set this to True only when running in Google Colab with data in Drive.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

# Environment variables make local and Google Drive layouts configurable.
DATA_DIR = Path(os.environ.get('COVIDX_DATA_DIR', ROOT / 'data/raw/covidx')).expanduser()
LABEL_DIR = Path(os.environ.get('COVIDX_LABEL_DIR', ROOT / 'data/raw')).expanduser()
TRAIN_LABELS = LABEL_DIR / 'train_COVIDx_CXR-3A.txt'
TEST_LABELS = LABEL_DIR / 'test_COVIDx_CXR-3A.txt'
METADATA = ROOT / 'data/processed/patients.csv'
VALIDATED = ROOT / 'data/processed/validated_metadata.csv'
REPORT = ROOT / 'artifacts/results/eda/data_quality_report.json'
print('Image directory:', DATA_DIR)
print('Label directory:', LABEL_DIR)

## 2. Verify prerequisites and build upstream metadata

In [ ]:
# The importer verifies the two pinned official label-file checksums.
required_paths = [DATA_DIR, TRAIN_LABELS, TEST_LABELS]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Follow docs/DATASET.md; missing: ' + ', '.join(missing_paths))
METADATA.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
if not METADATA.exists():
    subprocess.run([
        sys.executable, 'scripts/import_covidx_metadata.py',
        '--labels', str(TRAIN_LABELS), str(TEST_LABELS),
        '--data-dir', str(DATA_DIR), '--output', str(METADATA),
    ], check=True)
print('Metadata CSV:', METADATA)

## 3. Audit files without creating training splits

In [ ]:
from covid_xray.data import audit_metadata_file  # noqa: E402

# Each metadata row is retained even when its local image is missing or corrupt.
audited_data, quality_report = audit_metadata_file(
    METADATA, DATA_DIR, VALIDATED, REPORT
)
display(audited_data.head())
print(json.dumps(quality_report, indent=2))

## 4. Confirm the official boundary

Any patient, exact-image, or perceptual-hash match across upstream train/test partitions is a leakage risk for review. Phase 2 reports these risks but does not move, remove, or split records.

In [ ]:
partition_counts = audited_data.groupby(['original_partition', 'label']).size().unstack(fill_value=0)
display(partition_counts)
print('Patient overlap count:', quality_report['patient_overlap_count'])
print('Exact duplicate groups:', quality_report['exact_duplicate_groups'])
print('Perceptual match groups:', quality_report['perceptual_match_groups'])
print('No Phase 3 training split was created.')